# Forest Memory — Audio Audit

**BioSCape acoustic proxy analysis for ecosystem resilience**

> **Disclaimer:** All scores are acoustic proxies. They indicate relative differences in the soundscape between sites and seasons — not species counts, confirmed biodiversity, or validated ecological metrics.

In [ ]:
import sys
from pathlib import Path

ROOT = Path(".").resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
from scipy.signal import spectrogram
import warnings
warnings.filterwarnings("ignore")

from src.audio_signals import process_directory
from src.scoring import compute_scores, OUTPUT_COLS

AUDIO_ROOT = ROOT / "data" / "bioscape" / "audio"
OUT_CSV    = ROOT / "outputs" / "audio_features.csv"
OUT_CSV.parent.mkdir(exist_ok=True)

ROLE_ORDER  = ["healthy", "recovering", "disturbed", "wet_dry"]
ROLE_COLORS = {
    "healthy":    "#2d7a4f",
    "recovering": "#e8a020",
    "disturbed":  "#c0392b",
    "wet_dry":    "#2980b9",
}
print("Audio root:", AUDIO_ROOT)

## 1  Feature extraction

In [ ]:
rows = process_directory(AUDIO_ROOT)
raw_df = pd.DataFrame(rows)
print(f"Loaded {len(raw_df)} WAV files across {raw_df['role'].nunique()} roles")
raw_df.drop(columns=["path"], errors="ignore").head()

## 2  Missingness & feature summary

In [ ]:
display_df = raw_df.drop(columns=["path"], errors="ignore")
print("Missing values:")
print(display_df.isnull().sum())
print("\nFeature summary:")
display_df.describe().T.round(4)

## 3  Proxy scoring

In [ ]:
scored_df = compute_scores(raw_df)
out_cols = [c for c in OUTPUT_COLS if c in scored_df.columns]
result_df = scored_df[out_cols].copy()
result_df.to_csv(OUT_CSV, index=False)
print(f"Saved → {OUT_CSV}")
result_df

## 4  Per-role mean scores

In [ ]:
score_cols = [
    "bird_activity_proxy", "insect_activity_proxy_score",
    "human_disturbance_proxy", "acoustic_richness_score",
    "bioacoustic_vitality_score",
]
score_cols = [c for c in score_cols if c in scored_df.columns]

role_means = (
    scored_df.groupby("role")[score_cols]
    .mean()
    .reindex([r for r in ROLE_ORDER if r in scored_df["role"].unique()])
    .round(2)
)
print(role_means.to_string())

## 5  Waveforms — one representative WAV per role

In [ ]:
rep_wavs = (
    scored_df.sort_values("filename")
    .groupby("role")
    .first()
    .reset_index()[["role", "site_id", "filename"]]
)

fig, axes = plt.subplots(len(rep_wavs), 1, figsize=(14, 3 * len(rep_wavs)))
if len(rep_wavs) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, rep_wavs.iterrows()):
    wav_path = AUDIO_ROOT / row["role"] / row["filename"]
    y_raw, sr = sf.read(wav_path, dtype="float32", always_2d=False)
    y = y_raw.mean(axis=1) if y_raw.ndim > 1 else y_raw
    max_samples = 60 * sr
    y = y[:max_samples]
    times = np.arange(len(y)) / sr
    color = ROLE_COLORS.get(row["role"], "steelblue")
    ax.plot(times, y, linewidth=0.4, color=color, alpha=0.8)
    ax.set_title(f"{row['role']}  |  {row['site_id']}  |  {row['filename']}",
                 fontsize=9, loc="left")
    ax.set_ylabel("Amplitude")
    ax.set_xlabel("Time (s)")
    ax.set_xlim(0, times[-1])

plt.suptitle("Waveforms — first 60 s per representative WAV", fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(ROOT / "outputs" / "waveforms.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/waveforms.png")

## 6  Spectrograms — log-power spectrogram per role

In [ ]:
fig, axes = plt.subplots(len(rep_wavs), 1, figsize=(14, 3.5 * len(rep_wavs)))
if len(rep_wavs) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, rep_wavs.iterrows()):
    wav_path = AUDIO_ROOT / row["role"] / row["filename"]
    y_raw, sr = sf.read(wav_path, dtype="float32", always_2d=False)
    y = y_raw.mean(axis=1) if y_raw.ndim > 1 else y_raw
    y = y[:60 * sr]

    nperseg = min(1024, len(y) // 4)
    f, t, Sxx = spectrogram(y, fs=sr, nperseg=nperseg, noverlap=nperseg // 2)
    Sxx_db = 10 * np.log10(Sxx + 1e-12)

    im = ax.pcolormesh(t, f / 1000, Sxx_db, shading="gouraud",
                       cmap="inferno", vmin=-80, vmax=0)
    fig.colorbar(im, ax=ax, label="dB")
    ax.set_ylim(0, min(sr / 2000, 24))   # cap display at 24 kHz
    ax.set_ylabel("Freq (kHz)")
    ax.set_xlabel("Time (s)")
    ax.set_title(f"{row['role']}  |  {row['site_id']}", fontsize=9, loc="left")

plt.suptitle("Spectrograms — first 60 s per representative WAV", fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(ROOT / "outputs" / "spectrograms.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/spectrograms.png")

## 7  Score comparison bar chart

In [ ]:
roles_present = [r for r in ROLE_ORDER if r in role_means.index]
n_scores = len(score_cols)
x = np.arange(len(roles_present))
bar_width = 0.15

fig, ax = plt.subplots(figsize=(13, 6))

score_labels = {
    "bird_activity_proxy":         "Bird activity\n(proxy)",
    "insect_activity_proxy_score": "Insect activity\n(proxy)",
    "human_disturbance_proxy":     "Human disturbance\n(proxy)",
    "acoustic_richness_score":     "Acoustic richness\n(proxy)",
    "bioacoustic_vitality_score":  "Bioacoustic\nvitality (proxy)",
}
palette = ["#2d7a4f", "#e8a020", "#c0392b", "#2980b9", "#8e44ad"]

for i, col in enumerate(score_cols):
    vals = role_means.loc[roles_present, col].values
    offset = (i - n_scores / 2 + 0.5) * bar_width
    bars = ax.bar(x + offset, vals, bar_width,
                  label=score_labels.get(col, col),
                  color=palette[i % len(palette)], alpha=0.85, edgecolor="white")
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5, f"{val:.0f}",
                ha="center", va="bottom", fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels([r.replace("_", " ").title() for r in roles_present], fontsize=10)
ax.set_ylim(0, 115)
ax.set_ylabel("Proxy score (0–100, site-relative)", fontsize=10)
ax.set_title(
    "Acoustic Proxy Scores by Demo Site Role\n"
    "(relative within this 12-file sample — not absolute biodiversity measures)",
    fontsize=11,
)
ax.legend(loc="upper right", fontsize=8, ncol=2)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(ROOT / "outputs" / "score_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/score_comparison.png")

## 8  Narrative summary

In [ ]:
print("=" * 60)
print("ACOUSTIC PROXY SUMMARY — Forest Memory demo sites")
print("=" * 60)
for role in roles_present:
    row = role_means.loc[role]
    vitality = row.get("bioacoustic_vitality_score", float("nan"))
    richness  = row.get("acoustic_richness_score",   float("nan"))
    human     = row.get("human_disturbance_proxy",    float("nan"))
    print(f"\n  {role.upper().replace('_', ' ')}")
    print(f"    Vitality proxy : {vitality:.1f} / 100")
    print(f"    Richness proxy : {richness:.1f} / 100")
    print(f"    Human noise    : {human:.1f} / 100")
print()
print("NOTE: All values are relative within this 12-file sample.")
print("      Do not interpret as absolute ecosystem health scores.")